In [3]:
import pandas as pd
import re
import json
import ast
from os import fdopen, remove
from tempfile import mkstemp
from shutil import move, copymode
from pprint import pprint 
from io import StringIO
import os
import string
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def flatten(xss):
    return [x for xs in xss for x in xs]

### Daten veranschaulichen:
- Häufigkeit der KI Terme
- Häufigste Wörter
- TF-IDF Wörter Veranschaulichen
- length of sentences / number of clauses
- number / type of verbs
- number / type of adjektives

In [4]:
plots_dir = "plots/"
try:
    os.mkdir(plots_dir)
    print(f"Directory '{plots_dir}' created successfully.")
except FileExistsError:
    print(f"Directory '{plots_dir}' already exists.")

Directory 'plots/' already exists.


In [5]:
docs_ki = pd.read_json(StringIO(open('generated_data/KI_sentences_tfidf.json').read()), orient="index")

ki_docs = docs_ki[docs_ki['sentences'].isna() == False].reset_index()
non_ki_docs = docs_ki[docs_ki['sentences'].isna() == True].reset_index()

print(f"Number of KI docs: {len(ki_docs)}")
print(f"Number of other docs: {len(non_ki_docs)}")
non_ki_docs.head()

Number of KI docs: 0
Number of other docs: 152225


,index,sentences,top_words
0,6036703.txt,NaN,"['unterstützen', 'zwangshaft söder', 'genüssli..."
1,0224025.txt,NaN,"['zivilisten dramatischen', 'nkunda erschütter..."
2,5903496.txt,NaN,"['kommunist laufe', 'radikalisieren', 'kommuni..."
3,1184664.txt,NaN,"['andré', 'andré kunz', 'kunz', 'ishikawa prof..."
4,806049.txt,NaN,"['patienten fördern', 'patienten', 'gesundheit..."


In [6]:
docs_ai = pd.read_json(StringIO(open('generated_data/AI_sentences_tfidf.json').read()), orient="index")

ai_docs = docs_ai[docs_ai['sentences'].isna() == False].reset_index()
non_ai_docs = docs_ai[docs_ai['sentences'].isna() == True].reset_index()

print(f"Number of AI docs: {len(ai_docs)}")
print(f"Number of other docs: {len(non_ai_docs)}")
docs_ai.head()

Number of AI docs: 0
Number of other docs: 574


,sentences,top_words
5315432.txt,NaN,"['scooter 25', 'scooter', 'pushes scooter', 'p..."
5124643.txt,NaN,"['unarmed civilians', 'government forces', 'ba..."
5094180.txt,NaN,"['weiwei does', 'weiwei', 'shortcoming', 'nomi..."
851887.txt,NaN,"['philipp palm', 'philipp', 'palm crisis', 'pa..."
5455271.txt,NaN,"['world 2000', 'world', 'ki', '2000 ki', '2000']"


In [7]:
docs_sents = pd.read_csv(StringIO(open('generated_data/KI_sentences_translated.csv').read()))
docs_full = pd.read_csv(StringIO(open('generated_data/KI_sentences_full_translated.csv').read()))
docs_leipzig = pd.read_json(StringIO(open('generated_data/Leipzig_KI_sentences.json').read()), orient="index")

print(f"Number of leipzig 2024 docs: {len(docs_leipzig)}")
docs_full.head()

Number of leipzig 2024 docs: 3357


,id,text,translation
0,1739352_full.txt,Mit einer Mahnwache gegen die Todesstrafe und ...,With a vigil against the death penalty and a s...
1,1364384_full.txt,Berlin taz – Ein edleres Spendensiegel ist kau...,Berlin taz - A nobler seal of approval for don...
2,483913_full.txt,930 Friedrichstadtpalast Eu cand vreau sa flui...,930 Friedrichstadtpalast Eu cand vreau sa flui...
3,1305430_full.txt,München tazdpa – Die Mahnwache mahnte die Spru...,Munich tazdpa - The vigil warned the banners o...
4,1573221_full.txt,Berlin taz – Schon am kommenden Donnerstag wer...,"Berlin taz - The first refugees from Serbia, i..."


In [ ]:
# Häufigste Wörter insgesamt
def top_words(df_col, k):
    top_k = df_col.value_counts().head(k)
    print(f"\n{k} most frequent words:\n", top_k)
    return top_k

# TODO: Häufigste Wörter pro jahr 
def top_words_year(df, k):
    # TODO: find year, jahr aus id -> datei -> metadaten

    # group by year
    years = df.groupby(['year'])#.mean()
    # calc top words
    top_y = [(y[0],top_words(y[1],k)) for y in years]
    for y in top_y:
        print(f"\n{k} most frequent words:\n", y)
    return top_y

def get_POS(token_list):
    return [token.pos_ for token in token_list]

# Count top words by POS tag (supports token lists, spaCy Docs/Spans, or raw text)
def get_top_POS(df_col, pos, k):
    """Count the top-k words whose spaCy POS tag equals `pos`.

    Parameters
    ----------
    df_col : iterable
        A pandas Series / list-like containing either:
        - lists of spaCy Token objects
        - spaCy Doc/Span objects
        - raw text strings (the function will try to load a spaCy model)
    pos : str
        POS tag to filter for (e.g., 'VERB', 'NOUN', 'ADJ')
    k : int
        Number of top items to return

    Returns
    -------
    pandas.Series
        Top-k words (lemmas, lowercased) and their counts
    """
    from collections import Counter
    import pandas as pd

    # helper to obtain tokens from an element
    def _tokens_from_item(item, nlp):
        # spaCy Token lists / Docs / Spans 
        if hasattr(item, "pos_") or (hasattr(item, "__iter__") and not isinstance(item, str) and len(item) and hasattr(item[0], "pos_")):
            # item is a Token, Doc, Span, or list of Tokens
            return list(item)
        # raw string -> use spaCy to tokenize/pos-tag
        return list(nlp(str(item)))

    # detect whether we need to load a model (when any element is a string)
    try:
        # df_col can be a pandas Series
        iterable = df_col
        if hasattr(df_col, "dropna"):
            iterable = df_col.dropna().values
    except Exception:
        iterable = df_col

    counts = Counter()
    for item in iterable:
        tokens = _tokens_from_item(item, nlp)
        for token in tokens:
            # ensure token has pos_ and is alphabetic
            if getattr(token, "pos_", None) == pos and getattr(token, "is_alpha", False):
                lemma = getattr(token, "lemma_", token.text).lower()
                counts[lemma] += 1

    top_k = pd.Series(dict(counts.most_common(k)))
    print(f"\n{k} most frequent {pos}:\n", top_k)
    return top_k

#### TFIDF of KI_sentences filtered with regex

In [14]:
# compare KI_sentences and KI_sentences_full ids
ids_full = [x.replace("_full", "") for x in docs_full["id"].tolist()]
all_ids = docs_sents["id"].tolist() + ids_full
set_all_ids = set(all_ids)
set_all_og_ids = set(docs_sents["id"].tolist() + docs_full["id"].tolist())
non_repeated_ids = [x for x in set_all_ids if x not in ids_full] + [x for x in ids_full if x not in set_all_ids]
print(f"Number of ids: {len(all_ids)}, len unique ids: {len(set_all_ids)}, len non-repeated ids: {len(non_repeated_ids)}")
print("Non-repeated ids:", str(non_repeated_ids))
print("all original ids:", str(set_all_og_ids))

# separate tfidf of ai-sentences and others


# combine

# calc tfidf on result

# visualize, ideally against other tfidf(s)


Number of ids: 1640, len unique ids: 578, len non-repeated ids: 428
Non-repeated ids: ['1923030.txt', '5778990.txt', '5184580.txt', '0317472.txt', '0405973.txt', '5024842.txt', '5778473.txt', '731924.txt', '810137.txt', '5022627.txt', '1737254.txt', '0407571.txt', '5185491.txt', '5828787.txt', '5094180.txt', '5909029.txt', '0405985.txt', '5455271.txt', '654241.txt', '5317252.txt', '1569401.txt', '488911.txt', '5049593.txt', '5455586.txt', '1608779.txt', '5486405.txt', '1459861.txt', '1520366.txt', '481239.txt', '1250429.txt', '5148086.txt', '5896979.txt', '0402844.txt', '1854838.txt', '5093306.txt', '851652.txt', '5488306.txt', '5315735.txt', '488760.txt', '5907912.txt', '5092918.txt', '0316205.txt', '0311176.txt', '1183035.txt', '1247624.txt', '5193462.txt', '477818.txt', '5725480.txt', '0316706.txt', '5050404.txt', '1459696.txt', '0313723.txt', '5093213.txt', '5318710.txt', '5123854.txt', '5123711.txt', '489964.txt', '5907884.txt', '5124567.txt', '5452100.txt', '0240207.txt', '040341

#### KI_sentence stats

In [10]:
# Häufigste Wörter pro satz / text

# Häufigkeit der einzelnen KI Terme

# average length of sentences / number of clauses

In [11]:
# number and type of verbs

# number and type of adjektives